# Day 05: Advanced Transformations, Multi-Table Joins & Data Modeling
> **Part 5 of 10 in the PySpark Production Data Engineering Series**

[![PySpark](https://img.shields.io/badge/PySpark-3.5%2B%20%2F%204.0%2B-F26522?style=for-the-badge&logo=apachespark&logoColor=white)](https://spark.apache.org/)
[![Python](https://img.shields.io/badge/Python-3.10%2B-3776AB?style=for-the-badge&logo=python&logoColor=white)](https://python.org)
[![Status](https://img.shields.io/badge/Module-Day%2005%20of%2010-008080?style=for-the-badge)](../README.md)

---

## 🔗 Day 05: Advanced Transformations, Multi-Table Joins & Data Modeling
### 🎯 Learning Objectives:
1. Model complex relational star schemas in PySpark across Orders, Customers, and Products.
2. Master all relational join strategies: **Inner**, **Left Outer**, **Right Outer**, and **Full Outer**.
3. Resolve ambiguous column name collisions cleanly using DataFrame aliases (`orders.alias("o")`).
4. Implement complex business rules and branching logic using `F.when().otherwise()`.
5. Execute multi-dimensional aggregations combining `groupBy()` with multi-metric `.agg()`.
6. Conduct customer demographic and geographical spend analysis.
---


### ⚙️ Initialize SparkSession for Day 05


In [163]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

spark = (
    SparkSession.builder
    .appName("Day5_Transformations_Joins")
    .master("local[*]")
    .getOrCreate()
)

print("Spark started")

Spark started


In [164]:
customers_data = [
    (101, "Rahul", "Kolkata"),
    (102, "Amit", "Delhi"),
    (103, "Sneha", "Mumbai"),
    (104, "Priya", "Pune"),
    (105, "Arjun", "Bangalore"),
    (106, "Riya", "Chennai"),
    (107, "Karan", "Hyderabad"),
    (108, "Neha", "Kolkata")
]

customers = spark.createDataFrame(
    customers_data,
    ["customer_id", "customer_name", "city"]
)

customers.show()

+-----------+-------------+---------+
|customer_id|customer_name|     city|
+-----------+-------------+---------+
|        101|        Rahul|  Kolkata|
|        102|         Amit|    Delhi|
|        103|        Sneha|   Mumbai|
|        104|        Priya|     Pune|
|        105|        Arjun|Bangalore|
|        106|         Riya|  Chennai|
|        107|        Karan|Hyderabad|
|        108|         Neha|  Kolkata|
+-----------+-------------+---------+



In [165]:
orders_data = [
    (1001, 101, 501, 1, "2026-09-01"),
    (1002, 102, 502, 2, "2026-09-01"),
    (1003, 101, 503, 1, "2026-09-02"),
    (1004, 103, 504, 2, "2026-09-02"),
    (1005, 104, 501, 1, "2026-09-03"),
    (1006, 105, 502, 3, "2026-09-03"),
    (1007, 106, 503, 2, "2026-09-04"),
    (1008, 102, 505, 1, "2026-09-04"),
    (1009, 107, 504, 1, "2026-09-05"),
    (1010, 108, 501, 1, "2026-09-05"),
    (1011, 101, 505, 2, "2026-09-06"),
    (1012, 103, 502, 4, "2026-09-06")
]

orders = spark.createDataFrame(
    orders_data,
    ["order_id", "customer_id", "product_id", "quantity", "order_date"]
)

orders.show()

+--------+-----------+----------+--------+----------+
|order_id|customer_id|product_id|quantity|order_date|
+--------+-----------+----------+--------+----------+
|    1001|        101|       501|       1|2026-09-01|
|    1002|        102|       502|       2|2026-09-01|
|    1003|        101|       503|       1|2026-09-02|
|    1004|        103|       504|       2|2026-09-02|
|    1005|        104|       501|       1|2026-09-03|
|    1006|        105|       502|       3|2026-09-03|
|    1007|        106|       503|       2|2026-09-04|
|    1008|        102|       505|       1|2026-09-04|
|    1009|        107|       504|       1|2026-09-05|
|    1010|        108|       501|       1|2026-09-05|
|    1011|        101|       505|       2|2026-09-06|
|    1012|        103|       502|       4|2026-09-06|
+--------+-----------+----------+--------+----------+



In [166]:
customers.printSchema()
products.printSchema()
orders.printSchema()

root
 |-- customer_id: long (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)

root
 |-- product_id: long (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)

root
 |-- order_id: long (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- product_id: long (nullable = true)
 |-- quantity: long (nullable = true)
 |-- order_date: string (nullable = true)



In [167]:
orders.select(
    "order_id",
    "customer_id",
    "product_id",
    "quantity"
).show()

+--------+-----------+----------+--------+
|order_id|customer_id|product_id|quantity|
+--------+-----------+----------+--------+
|    1001|        101|       501|       1|
|    1002|        102|       502|       2|
|    1003|        101|       503|       1|
|    1004|        103|       504|       2|
|    1005|        104|       501|       1|
|    1006|        105|       502|       3|
|    1007|        106|       503|       2|
|    1008|        102|       505|       1|
|    1009|        107|       504|       1|
|    1010|        108|       501|       1|
|    1011|        101|       505|       2|
|    1012|        103|       502|       4|
+--------+-----------+----------+--------+



In [168]:
products_data = [
    (501, "Laptop", "Electronics", 55000),
    (502, "Mouse", "Accessories", 800),
    (503, "Keyboard", "Accessories", 1500),
    (504, "Monitor", "Electronics", 12000),
    (505, "Headphones", "Accessories", 2500)
]

products = spark.createDataFrame(
    products_data,
    ["product_id", "product_name", "category", "price"]
)

products.show()

+----------+------------+-----------+-----+
|product_id|product_name|   category|price|
+----------+------------+-----------+-----+
|       501|      Laptop|Electronics|55000|
|       502|       Mouse|Accessories|  800|
|       503|    Keyboard|Accessories| 1500|
|       504|     Monitor|Electronics|12000|
|       505|  Headphones|Accessories| 2500|
+----------+------------+-----------+-----+



In [169]:
order_product = orders.join(
    products,
    on="product_id",
    how="left"
)

order_product.show()
order_product.printSchema()

+----------+--------+-----------+--------+----------+------------+-----------+-----+
|product_id|order_id|customer_id|quantity|order_date|product_name|   category|price|
+----------+--------+-----------+--------+----------+------------+-----------+-----+
|       501|    1001|        101|       1|2026-09-01|      Laptop|Electronics|55000|
|       502|    1002|        102|       2|2026-09-01|       Mouse|Accessories|  800|
|       503|    1003|        101|       1|2026-09-02|    Keyboard|Accessories| 1500|
|       504|    1004|        103|       2|2026-09-02|     Monitor|Electronics|12000|
|       502|    1006|        105|       3|2026-09-03|       Mouse|Accessories|  800|
|       501|    1005|        104|       1|2026-09-03|      Laptop|Electronics|55000|
|       503|    1007|        106|       2|2026-09-04|    Keyboard|Accessories| 1500|
|       504|    1009|        107|       1|2026-09-05|     Monitor|Electronics|12000|
|       505|    1008|        102|       1|2026-09-04|  Headphones

In [170]:
order_product = order_product.withColumn(
    "revenue",
    col("quantity") * col("price")
)

order_product.show()

+----------+--------+-----------+--------+----------+------------+-----------+-----+-------+
|product_id|order_id|customer_id|quantity|order_date|product_name|   category|price|revenue|
+----------+--------+-----------+--------+----------+------------+-----------+-----+-------+
|       501|    1001|        101|       1|2026-09-01|      Laptop|Electronics|55000|  55000|
|       502|    1002|        102|       2|2026-09-01|       Mouse|Accessories|  800|   1600|
|       503|    1003|        101|       1|2026-09-02|    Keyboard|Accessories| 1500|   1500|
|       504|    1004|        103|       2|2026-09-02|     Monitor|Electronics|12000|  24000|
|       502|    1006|        105|       3|2026-09-03|       Mouse|Accessories|  800|   2400|
|       501|    1005|        104|       1|2026-09-03|      Laptop|Electronics|55000|  55000|
|       503|    1007|        106|       2|2026-09-04|    Keyboard|Accessories| 1500|   3000|
|       504|    1009|        107|       1|2026-09-05|     Monitor|Elec

In [171]:
#When and Otherwise
order_product = order_product.withColumn(
    "order_value_category",
    when(col("revenue") >= 50000, "High Value")
    .when(col("revenue") >= 10000, "Medium Value")
    .otherwise("Low Value")
)

In [172]:
order_product.select(
    "order_id",
    "revenue",
    "order_value_category"
).show()

+--------+-------+--------------------+
|order_id|revenue|order_value_category|
+--------+-------+--------------------+
|    1001|  55000|          High Value|
|    1002|   1600|           Low Value|
|    1003|   1500|           Low Value|
|    1004|  24000|        Medium Value|
|    1006|   2400|           Low Value|
|    1005|  55000|          High Value|
|    1007|   3000|           Low Value|
|    1009|  12000|        Medium Value|
|    1008|   2500|           Low Value|
|    1010|  55000|          High Value|
|    1012|   3200|           Low Value|
|    1011|   5000|           Low Value|
+--------+-------+--------------------+



In [173]:
#Filter
high_value_orders = order_product.filter(
    col("revenue") >= 50000
)

high_value_orders.show()

+----------+--------+-----------+--------+----------+------------+-----------+-----+-------+--------------------+
|product_id|order_id|customer_id|quantity|order_date|product_name|   category|price|revenue|order_value_category|
+----------+--------+-----------+--------+----------+------------+-----------+-----+-------+--------------------+
|       501|    1001|        101|       1|2026-09-01|      Laptop|Electronics|55000|  55000|          High Value|
|       501|    1005|        104|       1|2026-09-03|      Laptop|Electronics|55000|  55000|          High Value|
|       501|    1010|        108|       1|2026-09-05|      Laptop|Electronics|55000|  55000|          High Value|
+----------+--------+-----------+--------+----------+------------+-----------+-----+-------+--------------------+



In [174]:
full_orders = order_product.join(
    customers,
    on="customer_id",
    how="left"
)

In [175]:
full_orders.show()

+-----------+----------+--------+--------+----------+------------+-----------+-----+-------+--------------------+-------------+---------+
|customer_id|product_id|order_id|quantity|order_date|product_name|   category|price|revenue|order_value_category|customer_name|     city|
+-----------+----------+--------+--------+----------+------------+-----------+-----+-------+--------------------+-------------+---------+
|        107|       504|    1009|       1|2026-09-05|     Monitor|Electronics|12000|  12000|        Medium Value|        Karan|Hyderabad|
|        103|       504|    1004|       2|2026-09-02|     Monitor|Electronics|12000|  24000|        Medium Value|        Sneha|   Mumbai|
|        103|       502|    1012|       4|2026-09-06|       Mouse|Accessories|  800|   3200|           Low Value|        Sneha|   Mumbai|
|        104|       501|    1005|       1|2026-09-03|      Laptop|Electronics|55000|  55000|          High Value|        Priya|     Pune|
|        106|       503|    1007| 

In [176]:
final_orders = full_orders.select(
    "order_id",
    "order_date",
    "customer_id",
    "customer_name",
    "city",
    "product_id",
    "product_name",
    "category",
    "quantity",
    "price",
    "revenue",
    "order_value_category"
)

final_orders.show()

+--------+----------+-----------+-------------+---------+----------+------------+-----------+--------+-----+-------+--------------------+
|order_id|order_date|customer_id|customer_name|     city|product_id|product_name|   category|quantity|price|revenue|order_value_category|
+--------+----------+-----------+-------------+---------+----------+------------+-----------+--------+-----+-------+--------------------+
|    1001|2026-09-01|        101|        Rahul|  Kolkata|       501|      Laptop|Electronics|       1|55000|  55000|          High Value|
|    1002|2026-09-01|        102|         Amit|    Delhi|       502|       Mouse|Accessories|       2|  800|   1600|           Low Value|
|    1003|2026-09-02|        101|        Rahul|  Kolkata|       503|    Keyboard|Accessories|       1| 1500|   1500|           Low Value|
|    1004|2026-09-02|        103|        Sneha|   Mumbai|       504|     Monitor|Electronics|       2|12000|  24000|        Medium Value|
|    1006|2026-09-03|        105| 

In [177]:
#Aggregation
customer_revenue = (
    final_orders
    .groupBy(
        "customer_id",
        "customer_name"
    )
    .agg(
        sum("revenue").alias("total_revenue"),
        countDistinct("order_id").alias("total_orders"),
        sum("quantity").alias("total_items")
    )
    .orderBy(desc("total_revenue"))
)

customer_revenue.show()

+-----------+-------------+-------------+------------+-----------+
|customer_id|customer_name|total_revenue|total_orders|total_items|
+-----------+-------------+-------------+------------+-----------+
|        101|        Rahul|        61500|           3|          4|
|        104|        Priya|        55000|           1|          1|
|        108|         Neha|        55000|           1|          1|
|        103|        Sneha|        27200|           2|          6|
|        107|        Karan|        12000|           1|          1|
|        102|         Amit|         4100|           2|          3|
|        106|         Riya|         3000|           1|          2|
|        105|        Arjun|         2400|           1|          3|
+-----------+-------------+-------------+------------+-----------+



In [178]:
category_analysis = (
    final_orders
    .groupBy("category")
    .agg(
        sum("revenue").alias("total_revenue"),
        sum("quantity").alias("units_sold"),
        countDistinct("order_id").alias("orders")
    )
    .orderBy(desc("total_revenue"))
)

category_analysis.show()

+-----------+-------------+----------+------+
|   category|total_revenue|units_sold|orders|
+-----------+-------------+----------+------+
|Electronics|       201000|         6|     5|
|Accessories|        19200|        15|     7|
+-----------+-------------+----------+------+



In [179]:
overall_stats = final_orders.agg(
    sum("revenue").alias("total_revenue"),
    avg("revenue").alias("average_order_value"),
    min("revenue").alias("minimum_order"),
    max("revenue").alias("maximum_order"),
    countDistinct("order_id").alias("total_orders")
)

overall_stats.show()

+-------------+-------------------+-------------+-------------+------------+
|total_revenue|average_order_value|minimum_order|maximum_order|total_orders|
+-------------+-------------------+-------------+-------------+------------+
|       220200|            18350.0|         1500|        55000|          12|
+-------------+-------------------+-------------+-------------+------------+



In [180]:
#Conditional aggregation
final_orders.agg(
    count(
        when(
            col("order_value_category") == "High Value",
            True
        )
    ).alias("high_value_orders"),

    count(
        when(
            col("order_value_category") == "Medium Value",
            True
        )
    ).alias("medium_value_orders"),

    count(
        when(
            col("order_value_category") == "Low Value",
            True
        )
    ).alias("low_value_orders")
).show()

+-----------------+-------------------+----------------+
|high_value_orders|medium_value_orders|low_value_orders|
+-----------------+-------------------+----------------+
|                3|                  2|               7|
+-----------------+-------------------+----------------+



In [181]:
#City-level analysis
city_analysis = (
    final_orders
    .groupBy("city")
    .agg(
        sum("revenue").alias("revenue"),
        countDistinct("customer_id").alias("customers"),
        countDistinct("order_id").alias("orders")
    )
    .orderBy(desc("revenue"))
)

city_analysis.show()

+---------+-------+---------+------+
|     city|revenue|customers|orders|
+---------+-------+---------+------+
|  Kolkata| 116500|        2|     4|
|     Pune|  55000|        1|     1|
|   Mumbai|  27200|        1|     2|
|Hyderabad|  12000|        1|     1|
|    Delhi|   4100|        1|     2|
|  Chennai|   3000|        1|     1|
|Bangalore|   2400|        1|     1|
+---------+-------+---------+------+



In [182]:
customer_products = (
    final_orders
    .groupBy(
        "customer_id",
        "customer_name"
    )
    .agg(
        collect_list("product_name").alias("products")
    )
)

customer_products.show(truncate=False)

+-----------+-------------+------------------------------+
|customer_id|customer_name|products                      |
+-----------+-------------+------------------------------+
|102        |Amit         |[Mouse, Headphones]           |
|106        |Riya         |[Keyboard]                    |
|105        |Arjun        |[Mouse]                       |
|103        |Sneha        |[Monitor, Mouse]              |
|101        |Rahul        |[Laptop, Keyboard, Headphones]|
|104        |Priya        |[Laptop]                      |
|107        |Karan        |[Monitor]                     |
|108        |Neha         |[Laptop]                      |
+-----------+-------------+------------------------------+



In [183]:
customer_products_unique = (
    final_orders
    .groupBy(
        "customer_id",
        "customer_name"
    )
    .agg(
        collect_set("product_name").alias("unique_products")
    )
)

customer_products_unique.show(truncate=False)

+-----------+-------------+------------------------------+
|customer_id|customer_name|unique_products               |
+-----------+-------------+------------------------------+
|107        |Karan        |[Monitor]                     |
|103        |Sneha        |[Mouse, Monitor]              |
|104        |Priya        |[Laptop]                      |
|106        |Riya         |[Keyboard]                    |
|105        |Arjun        |[Mouse]                       |
|108        |Neha         |[Laptop]                      |
|101        |Rahul        |[Headphones, Keyboard, Laptop]|
|102        |Amit         |[Mouse, Headphones]           |
+-----------+-------------+------------------------------+



In [184]:
customer_test = spark.createDataFrame(
    [
        (101, "Rahul"),
        (102, "Amit"),
        (999, "Unknown Customer")
    ],
    ["customer_id", "name"]
)

In [185]:
#Inner Join
order_test = spark.createDataFrame(
    [
        (1, 101, 500),
        (2, 102, 700),
        (3, 103, 900)
    ],
    ["order_id", "customer_id", "amount"]
)

In [186]:
#Left Join
left = order_test.join(
    customer_test,
    "customer_id",
    "left"
)

left.show()

+-----------+--------+------+-----+
|customer_id|order_id|amount| name|
+-----------+--------+------+-----+
|        101|       1|   500|Rahul|
|        102|       2|   700| Amit|
|        103|       3|   900| NULL|
+-----------+--------+------+-----+



In [187]:
#Right Join
right = order_test.join(
    customer_test,
    "customer_id",
    "right"
)

right.show()

+-----------+--------+------+----------------+
|customer_id|order_id|amount|            name|
+-----------+--------+------+----------------+
|        101|       1|   500|           Rahul|
|        102|       2|   700|            Amit|
|        999|    NULL|  NULL|Unknown Customer|
+-----------+--------+------+----------------+



In [188]:
#Full Join
full = order_test.join(
    customer_test,
    "customer_id",
    "full"
)

full.show()

+-----------+--------+------+----------------+
|customer_id|order_id|amount|            name|
+-----------+--------+------+----------------+
|        101|       1|   500|           Rahul|
|        102|       2|   700|            Amit|
|        103|       3|   900|            NULL|
|        999|    NULL|  NULL|Unknown Customer|
+-----------+--------+------+----------------+



In [189]:
#Handle duplicate column names
orders.join(
    customers,
    orders.customer_id == customers.customer_id
)

DataFrame[order_id: bigint, customer_id: bigint, product_id: bigint, quantity: bigint, order_date: string, customer_id: bigint, customer_name: string, city: string]

In [190]:
o = orders.alias("o")
c = customers.alias("c")

joined = o.join(
    c,
    col("o.customer_id") == col("c.customer_id"),
    "left"
)

In [191]:
joined.select(
    col("o.order_id"),
    col("o.customer_id"),
    col("c.customer_name"),
    col("c.city")
).show()

+--------+-----------+-------------+---------+
|order_id|customer_id|customer_name|     city|
+--------+-----------+-------------+---------+
|    1001|        101|        Rahul|  Kolkata|
|    1003|        101|        Rahul|  Kolkata|
|    1002|        102|         Amit|    Delhi|
|    1004|        103|        Sneha|   Mumbai|
|    1005|        104|        Priya|     Pune|
|    1006|        105|        Arjun|Bangalore|
|    1007|        106|         Riya|  Chennai|
|    1009|        107|        Karan|Hyderabad|
|    1008|        102|         Amit|    Delhi|
|    1010|        108|         Neha|  Kolkata|
|    1012|        103|        Sneha|   Mumbai|
|    1011|        101|        Rahul|  Kolkata|
+--------+-----------+-------------+---------+



Most important thing today
Don't memorize syntax.
You should understand this flow:
                 RAW DATA
                    │
                    ▼
              select/filter
                    │
                    ▼
              withColumn()
                    │
                    ▼
                  JOIN
              ┌─────┴─────┐
              ▼           ▼
          customers    products
              │           │
              └─────┬─────┘
                    ▼
                groupBy()
                    │
                    ▼
                  agg()
                    │
                    ▼
                when()
                    │
                    ▼
              BUSINESS DATA


---
### 🎓 Day 05 Complete!
Great job completing this module!
👉 **Next Up:** [**Day 06: Analytical Window Functions Deep Dive**](./Day_06_Analytical_Window_Functions_Deep_Dive.ipynb)
